# Notebook 05 — Prompt-locked v35 full-run + v36/v37 analysis

Purpose:
1. Restore the exact old prompt style that produced the trusted `v35_fulltest_preds.json` on `generated_v4style_300`.
2. Run a **smoke-50 reproducibility check** before any expensive full-run.
3. If the smoke gate passes, optionally run full inference on:
   - `generated_v4style_300.json` as regression test
   - `benchmark_v2_1000.json` / v2.1 as balanced benchmark
4. Apply parsefix + v35 symbolic verifier.
5. Generate subset reports and v36/v37 analysis-only reports for MC and statement-form.

No MC/statement correction is applied in this notebook. Those remain analysis-only.

In [ ]:
# ============================================================
# 0. Configuration
# ============================================================
from pathlib import Path
import os, re, json, math, time, shutil, random
from collections import Counter, defaultdict

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Paths
BASE_MODEL = '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1'
ADAPTER = '/kaggle/input/datasets/yixuanisthebest/v2333333'
OLD_PRED_CANDIDATES = [
    '/kaggle/input/datasets/yixuanisthebest/v2333333/full_model_eval_v2_flatten_preds.json',
    '/kaggle/input/datasets/yixuanisthebest/v2333333/v35_fulltest_preds.json',
    '/kaggle/working/full_model_eval_v2_flatten_preds.json',
    '/mnt/data/full_model_eval_v2_flatten_preds.json',
]
DATASETS = {
    'generated_v4style_300': '/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json',
    # This path should point to v2.1 if the dataset file has been replaced/patched.
    'benchmark_v2_1000': '/kaggle/input/datasets/nguyenminhtric/test-pipeline/benchmark_v2_1000.json',
}

# Run controls
RUN_INFERENCE = True
SMOKE_DATASET = 'generated_v4style_300'
SMOKE_N = 50
RUN_FULL_AFTER_SMOKE_PASS = False  # Set True only after smoke passes.
MAX_SAMPLES_PER_DATASET = None     # Optional debug cap for full run.
CHECKPOINT_EVERY = 25

# Generation config. Keep deterministic.
MAX_NEW_TOKENS = 192
DO_SAMPLE = False

# Smoke gate: new smoke should not be worse than old artifact by more than this many correct answers.
SMOKE_ALLOWED_DROP_CORRECT = 3

print('OUT_DIR =', OUT_DIR)
print('RUN_INFERENCE =', RUN_INFERENCE)
print('RUN_FULL_AFTER_SMOKE_PASS =', RUN_FULL_AFTER_SMOKE_PASS)

In [ ]:
# ============================================================
# 1. IO + flatten + metrics utilities
# ============================================================
LABELS = ['A','B','C','D','Yes','No','Unknown']
MC_LABELS = ['A','B','C','D']
YNU_LABELS = ['Yes','No','Unknown']

FINAL_RE = re.compile(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)

def load_json(path, required=True):
    p = Path(path)
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            return json.load(f), p
    if required:
        raise FileNotFoundError(path)
    return None, None

def find_first(paths):
    for p in paths:
        if Path(p).exists(): return Path(p)
    return None

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    return path

def normalize_label(x):
    if x is None: return None
    s=str(x).strip()
    if not s: return None
    up=s.upper()
    if up in MC_LABELS: return up
    t=s.title()
    if t in YNU_LABELS: return t
    return None

def strict_parse_answer(text, allowed):
    """Trust explicit Final Answer only. If missing, use cautious Unknown heuristic for YNU only."""
    if not text:
        return 'UNPARSEABLE'
    hits = FINAL_RE.findall(str(text))
    if hits:
        lab = normalize_label(hits[-1])
        return lab if lab in allowed else 'UNPARSEABLE'
    low = str(text).lower()
    # Missing-final fallback: only for YNU and only for clearly indeterminate reasoning.
    if set(allowed) == set(YNU_LABELS):
        unknown_markers = [
            'cannot conclude', 'cannot determine', 'not enough information',
            'insufficient information', 'unknown whether', 'cannot infer',
            'not necessarily true', 'does not allow us to conclude',
            'there is no premise', 'no premise states', 'no premise provides'
        ]
        if any(m in low for m in unknown_markers):
            return 'Unknown'
    return 'UNPARSEABLE'

def flatten_records(raw, dataset_name):
    rows=[]
    for rec_i, rec in enumerate(raw):
        qs = rec.get('questions') or rec.get('question') or []
        ans = rec.get('answers') or rec.get('answer') or []
        exps = rec.get('explanation') or rec.get('explanations') or []
        idxs = rec.get('idx') or []
        if isinstance(qs, str): qs=[qs]
        if isinstance(ans, str): ans=[ans]
        if isinstance(exps, str): exps=[exps]*len(qs)
        if not isinstance(idxs, list): idxs=[]
        for q_i, q in enumerate(qs):
            gold = normalize_label(ans[q_i] if q_i < len(ans) else None)
            exp = exps[q_i] if isinstance(exps, list) and q_i < len(exps) else None
            idx = idxs[q_i] if isinstance(idxs, list) and q_i < len(idxs) else []
            is_mc = gold in MC_LABELS or bool(re.search(r'\n\s*A\.', str(q)))
            rows.append({
                'dataset': dataset_name,
                'flat_id': f'{dataset_name}:{rec_i}:{q_i}',
                'rec_i': rec_i,
                'q_i': q_i,
                'idx': idx,
                'premises_FOL': rec.get('premises-FOL') or rec.get('premises_FOL') or [],
                'premises_NL': rec.get('premises-NL') or rec.get('premises_NL') or [],
                'question': q,
                'gold': gold,
                'gold_raw': ans[q_i] if q_i < len(ans) else None,
                'reference_explanation': exp,
                'is_mc': bool(is_mc),
                'is_ynu': not bool(is_mc),
            })
    return rows

def per_label_metrics(rows, pred_key):
    out = {}
    for lab in LABELS:
        tp=sum(1 for r in rows if r.get('gold')==lab and r.get(pred_key)==lab)
        fp=sum(1 for r in rows if r.get('gold')!=lab and r.get(pred_key)==lab)
        fn=sum(1 for r in rows if r.get('gold')==lab and r.get(pred_key)!=lab)
        prec=tp/(tp+fp) if tp+fp else 0.0
        rec=tp/(tp+fn) if tp+fn else 0.0
        f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
        out[lab]={'precision':prec,'recall':rec,'f1':f1,'support':sum(1 for r in rows if r.get('gold')==lab),'pred_count':sum(1 for r in rows if r.get(pred_key)==lab)}
    return out

def metrics(rows, pred_key):
    n=len(rows)
    correct=sum(1 for r in rows if r.get(pred_key)==r.get('gold'))
    pl=per_label_metrics(rows, pred_key)
    macro7=sum(pl[l]['f1'] for l in LABELS)/len(LABELS)
    weighted=sum(pl[l]['f1']*pl[l]['support'] for l in LABELS)/max(1,sum(pl[l]['support'] for l in LABELS))
    mc_macro=sum(pl[l]['f1'] for l in MC_LABELS)/4
    ynu_macro=sum(pl[l]['f1'] for l in YNU_LABELS)/3
    return {
        'n': n, 'correct': correct, 'acc': correct/n if n else 0,
        'micro_f1': correct/n if n else 0, 'macro7': macro7,
        'weighted_f1': weighted, 'mc_macro': mc_macro, 'ynu_macro': ynu_macro,
        'per_label': pl,
        'pred_distribution': dict(Counter(r.get(pred_key,'MISSING') for r in rows)),
        'gold_distribution': dict(Counter(r.get('gold') for r in rows)),
    }

print('Utilities ready')

In [ ]:
# ============================================================
# 2. Prompt restoration
# ============================================================
old_pred_path = find_first(OLD_PRED_CANDIDATES)
old_prompt_by_key = {}
old_pred_by_key = {}
if old_pred_path:
    old_preds, _ = load_json(old_pred_path)
    print('Loaded old prompt/pred artifact:', old_pred_path, 'n=', len(old_preds))
    for r in old_preds:
        q = str(r.get('question','')).strip()
        key = re.sub(r'\s+',' ', q.lower())
        old_pred_by_key[key] = r
        p = r.get('prompt') or r.get('prompt_full') or r.get('prompt_text')
        # prompt_preview may be truncated; use it only as a weak fallback.
        if p:
            old_prompt_by_key[key] = p
else:
    print('WARNING: old prompt artifact not found. Will use canonical restored prompt.')

def canonical_prompt(row):
    allowed = 'A, B, C, or D' if row['is_mc'] else 'Yes, No, or Unknown'
    final_hint = '<A/B/C/D>' if row['is_mc'] else '<Yes/No/Unknown>'
    premises = row.get('premises_NL') or []
    ptxt = '\n'.join(f'{i+1}. {p}' for i,p in enumerate(premises))
    # Restored lines are mandatory.
    return (
        'You are solving a logic-based educational QA problem.\n\n'
        f'Premises:\n{ptxt}\n\n'
        f'Question:\n{row["question"]}\n\n'
        'Instructions:\n'
        '- Reason from the premises only.\n'
        '- Use only the given premises. Do not use outside knowledge.\n'
        f'- The final answer must be one of: {allowed}.\n'
        '- Answer only the logical truth of the question from the premises.\n'
        '- End with one line: Final Answer: X.\n'
    )

def make_prompt(row):
    key = re.sub(r'\s+',' ', str(row.get('question','')).strip().lower())
    # Exact artifact prompt if available. Otherwise canonical restored prompt.
    return old_prompt_by_key.get(key) or canonical_prompt(row)

def has_final_answer_guard(prompt):
    p = str(prompt)
    # Accept either the canonical rebuilt line or the exact old artifact style.
    return bool(
        re.search(r'End with one line:\s*Final Answer\s*:', p, flags=re.I)
        or re.search(r'End with:\s*Final Answer\s*:', p, flags=re.I)
        or re.search(r'Final Answer\s*:\s*(?:X|<[^>]+>|Yes|No|Unknown|A|B|C|D)', p, flags=re.I)
    )

# Prompt sanity: restored lines must exist, but final-answer placeholder may be X or <labels>.
# Prompt sanity: restored lines must exist, but final-answer placeholder may be X or <labels>.
def prompt_sanity(rows, n=20):
    sample = rows[:n]
    for r in sample:
        p = make_prompt(r)
        assert 'Use only the given premises. Do not use outside knowledge.' in p, 'missing outside-knowledge guard'
        assert has_final_answer_guard(p), 'missing final-answer line guard'
    print(f'Prompt sanity passed on {len(sample)} rows. old exact prompts available for {len(old_prompt_by_key)} questions.')

In [ ]:
# ============================================================
# 3. v35 symbolic engine: typed PROVE_NO/E1 only
# ============================================================
PRED_RE = re.compile(r'¬?([A-Z][A-Za-z0-9_]*)\(x\)')
IMP_RE = re.compile(r'∀x\s*\((¬?[A-Z][A-Za-z0-9_]*\(x\))\s*→\s*(¬?[A-Z][A-Za-z0-9_]*\(x\))\)')
UNIV_RE = re.compile(r'∀x\s*\((¬?[A-Z][A-Za-z0-9_]*)\(x\)\)')
EXIST_RE = re.compile(r'∃x\s*\((¬?[A-Z][A-Za-z0-9_]*)\(x\)\)')

def atom_name(a):
    return a.replace('(x)','').strip()

def tokenize_words(s):
    return set(re.findall(r'[a-z]+', str(s).lower()))

def pred_to_words(pred):
    parts = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', pred)
    return set(p.lower() for p in parts if len(p)>1)

def best_predicate_match(question, predicates):
    qwords = tokenize_words(question)
    # remove generic words
    stop = {'does','do','all','at','least','one','some','is','there','who','which','statement','true','based','on','the','above','premises','according','following','if','then','no','every','a','an','can','we','conclude'}
    qwords = {w for w in qwords if w not in stop}
    best=(None,0,set())
    for p in predicates:
        pw = pred_to_words(p)
        if not pw: continue
        score = len(qwords & pw)/len(pw)
        if score > best[1]: best=(p,score,pw)
    return best

def build_closure(fol):
    pos=set(); neg=set(); edges=[]
    all_preds=set()
    for s in fol:
        s=str(s).strip()
        m=UNIV_RE.match(s)
        if m:
            a=m.group(1)
            name=a[1:] if a.startswith('¬') else a
            all_preds.add(name)
            if a.startswith('¬'): neg.add(name)
            else: pos.add(name)
        m=EXIST_RE.match(s)
        if m:
            a=m.group(1); name=a[1:] if a.startswith('¬') else a
            all_preds.add(name)
        m=IMP_RE.match(s)
        if m:
            left=atom_name(m.group(1)); right=atom_name(m.group(2))
            lneg=left.startswith('¬'); rneg=right.startswith('¬')
            lp=left[1:] if lneg else left; rp=right[1:] if rneg else right
            all_preds.update([lp,rp])
            edges.append((lp,lneg,rp,rneg, s))
            # contrapositive: A->B gives ¬B->¬A, including signs.
            edges.append((rp,not rneg,lp,not lneg, 'CONTRAPOSITIVE_OF: '+s))
    changed=True; path_pos={p:'GIVEN ∀x '+p for p in pos}; path_neg={p:'GIVEN ∀x ¬'+p for p in neg}
    while changed:
        changed=False
        for lp,lneg,rp,rneg,src in edges:
            has_left = lp in (neg if lneg else pos)
            if not has_left: continue
            dest_set = neg if rneg else pos
            path_set = path_neg if rneg else path_pos
            src_path = path_neg.get(lp) if lneg else path_pos.get(lp)
            if rp not in dest_set:
                dest_set.add(rp); path_set[rp]=src if not src_path else f'{src_path} -> {src}'; changed=True
    return {'pos':pos,'neg':neg,'predicates':all_preds,'path_pos':path_pos,'path_neg':path_neg}

def question_type(q):
    low=str(q).lower()
    if re.search(r'at least one|is there some|does at least one|some ', low): return 'existential'
    if re.search(r'\ball\b|\bevery\b|do all|does all|are all|is it true that every', low): return 'universal'
    if 'statement:' in low and ' if ' in low and ' then ' in low: return 'statement_conditional'
    return 'other'

def v35_verdict(row):
    if not row.get('is_ynu'): return ('ABSTAIN', None)
    q = row.get('question','')
    qt = question_type(q)
    closure = build_closure(row.get('premises_FOL') or [])
    target, score, tw = best_predicate_match(q, closure['predicates'])
    cert = {'qtype':qt,'target':target,'target_score':score,'target_words':sorted(tw) if tw else [],
            'neg_proof_universal':False,'pos_proof_universal':False,'neg_proof_path':None,'pos_proof_path':None}
    if not target or score < 0.5:
        cert['reason']='NO_TARGET_MATCH'; return ('ABSTAIN', cert)
    neg = target in closure['neg']; pos = target in closure['pos']
    cert.update({'neg_proof_universal':neg,'pos_proof_universal':pos,
                 'neg_proof_path':closure['path_neg'].get(target), 'pos_proof_path':closure['path_pos'].get(target)})
    # E1: for existential question, universal negative proves No even under positive conflict by dataset convention/classical target.
    if qt == 'existential' and neg:
        cert['rule']='E1_FORALL_NEG_NOT_EXISTS'; return ('PROVE_NO', cert)
    # Universal negative: if asked universal/other and we prove no target, No; but abstain if positive conflict.
    if qt in ('universal','other') and neg and not pos:
        cert['rule']='TYPED_PROVE_NO_NO_POSITIVE_CONFLICT'; return ('PROVE_NO', cert)
    cert['reason']='NO_SAFE_PROVE_NO'; return ('ABSTAIN', cert)

def apply_parsefix_and_v35(rows):
    for r in rows:
        allowed = MC_LABELS if r['is_mc'] else YNU_LABELS
        r['pred_raw'] = strict_parse_answer(r.get('raw_output',''), allowed)
        r['pred_parsefix'] = r['pred_raw']
        r['pred_v35'] = r['pred_parsefix']
        r['v35_verdict'], r['v35_certificate'] = ('ABSTAIN', None)
        r['v35_changed'] = False
        if r['is_ynu'] and r['pred_parsefix'] != 'No':
            verdict, cert = v35_verdict(r)
            r['v35_verdict'], r['v35_certificate'] = verdict, cert
            if verdict == 'PROVE_NO':
                r['pred_v35'] = 'No'; r['v35_changed'] = True
    return rows

print('v35 symbolic engine ready')

In [ ]:
# ============================================================
# 4. Model loading + inference
# ============================================================
model = None; tok = None

def load_model_once():
    global model, tok
    if model is not None and tok is not None: return model, tok
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    print('Loading tokenizer:', ADAPTER)
    tok = AutoTokenizer.from_pretrained(ADAPTER, trust_remote_code=True)
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print('Loading base model:', BASE_MODEL, 'dtype=', dtype, 'cuda=', torch.cuda.is_available())
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype, device_map='auto' if torch.cuda.is_available() else None, trust_remote_code=True)
    print('Loading LoRA adapter:', ADAPTER)
    model = PeftModel.from_pretrained(base, ADAPTER)
    model.eval()
    return model, tok

def generate_one(prompt):
    import torch
    m,t = load_model_once()
    inputs = t(prompt, return_tensors='pt')
    inputs = {k:v.to(m.device) for k,v in inputs.items()}
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE, pad_token_id=t.eos_token_id, eos_token_id=t.eos_token_id)
    gen = out[0][inputs['input_ids'].shape[1]:]
    return t.decode(gen, skip_special_tokens=True)

print('Model loader ready')

In [ ]:
# ============================================================
# 5. Dataset runner + smoke gate
# ============================================================
def load_dataset_rows(name):
    raw, p = load_json(DATASETS[name])
    rows = flatten_records(raw, name)
    print(name, 'flattened:', len(rows), 'gold:', Counter(r['gold'] for r in rows), 'MC:', sum(r['is_mc'] for r in rows))
    prompt_sanity(rows)
    return rows

def attach_old_for_smoke(rows):
    for r in rows:
        key = re.sub(r'\s+',' ', str(r.get('question','')).strip().lower())
        old = old_pred_by_key.get(key)
        if old:
            # support multiple old prediction field names
            r['old_pred'] = old.get('pred_v35') or old.get('pred_selected') or old.get('pred_v32') or old.get('pred') or old.get('prediction')
        else:
            r['old_pred'] = None
    return rows

def run_inference_rows(rows, dataset_name, suffix, max_samples=None):
    out_path = OUT_DIR / f'05_{dataset_name}_{suffix}_preds.json'
    work = rows[:max_samples] if max_samples else rows
    # Resume if exact output exists and length matches
    if out_path.exists():
        saved, _ = load_json(out_path)
        if len(saved) == len(work):
            print('Reusing existing:', out_path)
            return saved
    result=[]
    start=time.time()
    for i, r in enumerate(work):
        rr=dict(r)
        rr['prompt'] = make_prompt(rr)
        if RUN_INFERENCE:
            try:
                t0=time.time(); rr['raw_output']=generate_one(rr['prompt']); rr['latency_sec']=time.time()-t0; rr['generation_error']=None
            except Exception as e:
                rr['raw_output']=''; rr['latency_sec']=None; rr['generation_error']=repr(e)
        else:
            rr['raw_output']=''; rr['latency_sec']=None; rr['generation_error']='RUN_INFERENCE_FALSE'
        result.append(rr)
        if (i+1)%CHECKPOINT_EVERY==0 or i+1==len(work):
            save_json(result, out_path)
            print(f'checkpoint {dataset_name}/{suffix}: {i+1}/{len(work)} elapsed={(time.time()-start)/60:.1f}m')
    return result

def evaluate_and_save(rows, dataset_name, suffix):
    rows = apply_parsefix_and_v35(rows)
    summary = {
        'version': 'notebook_05_prompt_locked_v35',
        'dataset': dataset_name,
        'suffix': suffix,
        'n': len(rows), 'n_mc': sum(r['is_mc'] for r in rows), 'n_ynu': sum(r['is_ynu'] for r in rows),
        'metrics_raw': metrics(rows, 'pred_raw'),
        'metrics_parsefix': metrics(rows, 'pred_parsefix'),
        'metrics_v35': metrics(rows, 'pred_v35'),
        'v35_flips': sum(1 for r in rows if r.get('v35_changed')),
        'v35_good_flips': sum(1 for r in rows if r.get('v35_changed') and r.get('pred_v35')==r.get('gold')),
        'v35_bad_flips': sum(1 for r in rows if r.get('v35_changed') and r.get('pred_v35')!=r.get('gold')),
        'missing_final_answer': sum(1 for r in rows if not FINAL_RE.findall(str(r.get('raw_output','')))),
        'generation_errors': sum(1 for r in rows if r.get('generation_error')),
    }
    pred_path = save_json(rows, OUT_DIR / f'05_{dataset_name}_{suffix}_preds_evaluated.json')
    summary_path = save_json(summary, OUT_DIR / f'05_{dataset_name}_{suffix}_summary.json')
    # Reload verify
    reloaded, _ = load_json(pred_path)
    assert len(reloaded)==len(rows), 'reload length mismatch'
    m2 = metrics(reloaded, 'pred_v35')
    assert abs(m2['acc'] - summary['metrics_v35']['acc']) < 1e-12, 'reload metric mismatch'
    print('Saved', pred_path)
    print('Saved', summary_path)
    print('v35 acc/macro7:', summary['metrics_v35']['acc'], summary['metrics_v35']['macro7'], 'flips:', summary['v35_flips'], 'good/bad:', summary['v35_good_flips'], summary['v35_bad_flips'])
    return rows, summary

# Smoke run
smoke_rows = load_dataset_rows(SMOKE_DATASET)
smoke_rows = attach_old_for_smoke(smoke_rows)
smoke_inferred = run_inference_rows(smoke_rows, SMOKE_DATASET, f'smoke{SMOKE_N}', max_samples=SMOKE_N)
smoke_eval, smoke_summary = evaluate_and_save(smoke_inferred, SMOKE_DATASET, f'smoke{SMOKE_N}')

# Compare against old artifact where possible
old_comparable = [r for r in smoke_eval if r.get('old_pred') in LABELS]
old_correct = sum(1 for r in old_comparable if r.get('old_pred') == r.get('gold'))
new_correct = sum(1 for r in old_comparable if r.get('pred_v35') == r.get('gold'))
smoke_pass = (not old_comparable) or (new_correct >= old_correct - SMOKE_ALLOWED_DROP_CORRECT)
smoke_gate = {
    'dataset': SMOKE_DATASET,
    'n': len(smoke_eval),
    'old_comparable': len(old_comparable),
    'old_correct': old_correct,
    'new_correct': new_correct,
    'allowed_drop': SMOKE_ALLOWED_DROP_CORRECT,
    'pass': smoke_pass,
    'new_metrics': smoke_summary['metrics_v35'],
}
save_json(smoke_gate, OUT_DIR / '05_smoke_gate.json')
print('SMOKE GATE:', smoke_gate)
if not smoke_pass:
    raise AssertionError('Smoke gate FAILED. Do not run full inference until prompt/generation drift is fixed.')

In [ ]:
# ============================================================
# 6. Optional full-run on both datasets after smoke passes
# ============================================================
full_summaries = {'smoke_gate': smoke_gate, 'datasets': {}}
if RUN_FULL_AFTER_SMOKE_PASS:
    for name in DATASETS:
        rows = load_dataset_rows(name)
        cap = MAX_SAMPLES_PER_DATASET
        inferred = run_inference_rows(rows, name, 'full', max_samples=cap)
        eval_rows, summ = evaluate_and_save(inferred, name, 'full')
        full_summaries['datasets'][name] = summ
else:
    print('RUN_FULL_AFTER_SMOKE_PASS=False, skipping expensive full-run. Set True after smoke passes.')

save_json(full_summaries, OUT_DIR / '05_combined_summary.json')
print('Combined summary saved:', OUT_DIR / '05_combined_summary.json')

In [ ]:
# ============================================================
# 7. Subset report + v36/v37 analysis-only helpers
# ============================================================
def has_direct_conflict(row):
    c = build_closure(row.get('premises_FOL') or [])
    return bool(c['pos'] & c['neg'])

def grammar_flags(q):
    flags=[]; s=str(q)
    if re.search(r'\ba (intern|exam|athlete|analyst|employee|applicant)\b', s, re.I): flags.append('ARTICLE_A_AN')
    if re.search(r'Do all \w+s? \w+\?', s): flags.append('POSSIBLY_UNNATURAL_QUESTION')
    return flags

def tag_rows(rows):
    for r in rows:
        r['question_type'] = question_type(r.get('question',''))
        r['has_direct_fol_conflict'] = has_direct_conflict(r)
        r['grammar_flags'] = grammar_flags(r.get('question',''))
        r['is_grammar_noisy'] = bool(r['grammar_flags'])
    return rows

def subset_metrics(rows, pred_key='pred_v35'):
    subsets = {
        'all': rows,
        'mc': [r for r in rows if r.get('is_mc')],
        'ynu': [r for r in rows if r.get('is_ynu')],
        'conflict': [r for r in rows if r.get('has_direct_fol_conflict')],
        'clean_no_conflict': [r for r in rows if not r.get('has_direct_fol_conflict')],
        'grammar_noisy': [r for r in rows if r.get('is_grammar_noisy')],
        'existential': [r for r in rows if r.get('question_type')=='existential'],
        'universal': [r for r in rows if r.get('question_type')=='universal'],
        'statement_conditional': [r for r in rows if r.get('question_type')=='statement_conditional'],
    }
    return {k: metrics(v, pred_key) if v else {'n':0} for k,v in subsets.items()}

# Simple MC/statement analysis-only placeholders: report candidates but never apply.
def mc_analysis_stub(rows):
    mc=[r for r in rows if r.get('is_mc')]
    # We intentionally do not apply. This stub counts errors by gold/pred to guide v36.
    errors=[r for r in mc if r.get('pred_v35') != r.get('gold')]
    return {
        'mode':'ANALYSIS_ONLY', 'n':len(mc), 'errors':len(errors),
        'error_gold_distribution': dict(Counter(r.get('gold') for r in errors)),
        'error_pred_distribution': dict(Counter(r.get('pred_v35') for r in errors)),
        'note':'MC correction disabled. Build polarity-aware unique-entailment prover separately; require >=0.9 precision on generated and benchmark v2.1 before apply.'
    }

def statement_analysis_stub(rows):
    st=[r for r in rows if r.get('question_type')=='statement_conditional']
    errors=[r for r in st if r.get('pred_v35') != r.get('gold')]
    return {
        'mode':'ANALYSIS_ONLY', 'n':len(st), 'errors':len(errors),
        'error_gold_distribution': dict(Counter(r.get('gold') for r in errors)),
        'error_pred_distribution': dict(Counter(r.get('pred_v35') for r in errors)),
        'note':'Statement correction disabled. Need antecedent->consequent proof and counterexample witness support before apply.'
    }

analysis_summary={}
# Analyze any evaluated full/smoke files produced by this notebook.
for path in sorted(OUT_DIR.glob('05_*_preds_evaluated.json')):
    rows,_=load_json(path)
    rows=tag_rows(rows)
    name=path.stem.replace('05_','').replace('_preds_evaluated','')
    analysis_summary[name]={
        'subset_metrics': subset_metrics(rows, 'pred_v35'),
        'mc_analysis': mc_analysis_stub(rows),
        'statement_analysis': statement_analysis_stub(rows),
    }
    save_json(analysis_summary[name], OUT_DIR / f'05_{name}_subset_and_analysis.json')

save_json(analysis_summary, OUT_DIR / '05_combined_subset_and_analysis.json')
print('Saved analysis summaries:', OUT_DIR / '05_combined_subset_and_analysis.json')

In [ ]:
# ============================================================
# 8. Risk report
# ============================================================
risk = {
    'version': 'notebook_05_prompt_locked_v35_fullrun_v36_analysis',
    'artifact_risk': 'LOW_RELOADED_SAVED_PREDS_WITH_CHECKPOINTS',
    'overfit_risk': 'LOW_SYMBOLIC_RULES_NO_SWEEP_PROMPT_LOCKED',
    'underfit_risk': 'MC_AND_STATEMENT_REMAIN_UNAPPLIED; GENERATION_QUALITY_AND_PROMPT_REPRODUCIBILITY_MUST_BE_MONITORED',
    'label_collapse': False,
    'prompt_lock': {
        'required_lines': [
            'Use only the given premises. Do not use outside knowledge.',
            'End with one line: Final Answer: <label>.'
        ],
        'old_prompt_artifact': str(old_pred_path) if old_pred_path else None,
        'old_exact_prompt_count': len(old_prompt_by_key),
    },
    'smoke_gate': smoke_gate,
    'correction_policy': {
        'v35_ynu_prove_no': 'APPLY_ONLY_WITH_PROOF_CERTIFICATE',
        'mc_option_proof': 'ANALYSIS_ONLY_DISABLED',
        'statement_form': 'ANALYSIS_ONLY_DISABLED',
    }
}
save_json(risk, OUT_DIR / '05_risk_report.json')
print(json.dumps(risk, indent=2)[:4000])